# 05 — Propulsion and Endurance

Cruise power from the drag polar, converted to pack current, then fed to
the **same** Peukert endurance model the multirotor uses
(`uavdesign.endurance`). Only the power-required front end differs
between platforms.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "flying_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "flying_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
with open(REPO_ROOT / "mission" / "mission_params.yaml") as f:
    mission = yaml.safe_load(f)
with open(PLATFORM_DIR / "design_point.yaml") as f:
    design = yaml.safe_load(f)
with open(PLATFORM_DIR / "airframe_params.yaml") as f:
    airframe = yaml.safe_load(f)

from uavdesign.atmosphere import G, isa_density, wing_loading_g_dm2, wing_loading_n_m2
from uavdesign.aero import Wing, DragPolar, FixedWingAero

alt_m = mission["mission"]["altitude_asl_m"]
rho = isa_density(alt_m, mission["environment"]["temperature_offset_c"])
wing = Wing.from_dict(airframe["wing"])
aero = design["aero"]
perf = design["performance"]
mass_kg = design["design_point"]["target_auw_kg"]
static_margin = design["design_point"]["static_margin"]
polar = DragPolar.from_oswald(aero["cd0"], wing.aspect_ratio, aero["oswald_e"], aero["cl_max"])

print(f"Site: {alt_m} m ASL, rho = {rho:.4f} kg/m^3")
print(f"Wing: S = {wing.area_dm2:.2f} dm^2, b = {wing.span_mm:.0f} mm, AR = {wing.aspect_ratio:.2f}")
print(f"Polar: CD0 = {polar.cd0}, k = {polar.k:.4f}, CLmax = {polar.cl_max}")


In [ ]:
from uavdesign import propulsion as prop, endurance as ed
from uavdesign.mass import MassBudget

pp, batt = design["propulsion"], design["battery"]
pack_v = batt["cells_series"] * batt["nominal_cell_v"]
battery = ed.Battery(
    capacity_ah=batt["capacity_ah"],
    nominal_voltage_v=pack_v,
    peukert_exponent=batt["peukert_exponent"],
    usable_fraction=batt["usable_fraction"],
)

# Mass from the shared budget (same build as notebook 03)
mb = MassBudget.from_yaml(
    PLATFORM_DIR / "airframe_params.yaml",
    REPO_ROOT / "payload" / "payload_manifest.yaml",
    active_payloads=design["payloads"],
    include_always_on=False,
)
mass_without_battery_g = mb.dry_mass_g
auw_kg = (mass_without_battery_g + batt["mass_g"]) * (1 + mb.contingency_fraction) / 1000
ac = FixedWingAero(auw_kg, wing.area_m2, polar, rho)

def electrical_w(aircraft, v, climb_rate=0.0):
    """Pack-side power: drag (+ climb) -> shaft -> electrical, plus avionics."""
    shaft = prop.climb_shaft_power_w(
        float(aircraft.drag_n(v)), v, aircraft.weight_n, climb_rate, pp["prop_efficiency"])
    return prop.electrical_power_w(shaft, pp["motor_efficiency"], pp["esc_efficiency"]) \
        + pp["avionics_power_w"]

print(f"AUW from mass budget: {auw_kg*1000:.0f} g, pack {batt['cells_series']}S "
      f"{pack_v:.1f} V {batt['capacity_ah']} Ah ({battery.energy_wh:.0f} Wh)")


## Operating points


In [ ]:
import pandas as pd

vs = ac.stall_speed()
points = {
    "best endurance": (max(ac.v_min_power, 1.2 * vs), 0.0),
    "cruise": (perf["cruise_speed_m_s"], 0.0),
    "climb": (perf["climb_speed_m_s"], perf["climb_rate_m_s"]),
}
rows = []
for name, (v, roc) in points.items():
    p = electrical_w(ac, v, roc)
    i = ed.pack_current_a(p, pack_v)
    rows.append({
        "point": name, "airspeed_m_s": v, "climb_m_s": roc,
        "electrical_w": p, "pack_current_a": i, "c_rate": i / batt["capacity_ah"],
        "endurance_min": ed.endurance_min(battery, i),
    })
ops = pd.DataFrame(rows).set_index("point")
ops.round(2)


## Motor and ESC sizing hint

Size for the climb point with headroom; the cruise point decides the
prop pitch (match pitch speed to roughly 1.3-1.5x cruise speed).


In [ ]:
p_climb_elec = ops.loc["climb", "electrical_w"] - pp["avionics_power_w"]
print(f"Climb electrical power (propulsion): {p_climb_elec:.0f} W")
print(f"Motor continuous rating >= {1.5 * p_climb_elec:.0f} W (1.5x headroom)")
print(f"ESC continuous current  >= {1.3 * p_climb_elec / pack_v:.0f} A (1.3x headroom)")
print(f"Prop pitch speed target ~ {1.4 * perf['cruise_speed_m_s']:.0f} m/s")


## Mission profile

Climb to working height, then cruise. Segments at different currents
share one pack via the Peukert-equivalent capacity (shared model).


In [ ]:
climb_height_m = 120.0
v_cr, v_cl, roc = perf["cruise_speed_m_s"], perf["climb_speed_m_s"], perf["climb_rate_m_s"]
segments = [
    ed.MissionSegment("climb", climb_height_m / roc, electrical_w(ac, v_cl, roc)),
    ed.MissionSegment("cruise", perf["target_endurance_min"] * 60, electrical_w(ac, v_cr)),
]
margin = ed.mission_margin(battery, segments)
print(f"Mission: climb {climb_height_m:.0f} m, cruise {perf['target_endurance_min']} min @ {v_cr} m/s")
print(f"Usable capacity left after mission: {margin:.0%} "
      f"({'OK' if margin >= 0 else 'SHORT'}; reserve of {1 - batt['usable_fraction']:.0%} still untouched)")


## Endurance vs battery capacity

Unlike the hover sweep, a heavier pack raises wing loading and therefore
drag *and* stall speed. Each capacity is re-flown at its own mass; the
red region is where stall speed exceeds the limit from `design_point.yaml`.


In [ ]:
pack_wh_per_g = battery.energy_wh / batt["mass_g"]   # baseline pack, incl. wiring
print(f"Baseline pack specific energy: {pack_wh_per_g*1000:.0f} Wh/kg")

caps = np.linspace(*batt["capacity_sweep_ah"], 40)
e_cruise, e_best, vs_arr = [], [], []
for c in caps:
    m_batt_g = c * pack_v / pack_wh_per_g
    m_kg = (mass_without_battery_g + m_batt_g) * (1 + mb.contingency_fraction) / 1000
    a = FixedWingAero(m_kg, wing.area_m2, polar, rho)
    b = ed.Battery(c, pack_v, batt["peukert_exponent"], usable_fraction=batt["usable_fraction"])
    v_best = max(a.v_min_power, 1.2 * a.stall_speed())
    e_cruise.append(ed.endurance_from_power_min(b, electrical_w(a, max(v_cr, 1.2 * a.stall_speed()))))
    e_best.append(ed.endurance_from_power_min(b, electrical_w(a, v_best)))
    vs_arr.append(a.stall_speed())

vs_arr = np.array(vs_arr)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(caps, e_best, label="at best-endurance speed")
ax.plot(caps, e_cruise, label=f"at cruise {v_cr} m/s (or 1.2 Vs if higher)")
ax.axhline(perf["target_endurance_min"], color="grey", ls="--", lw=1, label="target")
bad = vs_arr > perf["max_stall_speed_m_s"]
if bad.any():
    ax.axvspan(caps[bad].min(), caps[-1], color="C3", alpha=0.12, label="stall speed over limit")
ax.axvline(batt["capacity_ah"], color="k", lw=0.8, ls=":")
ax.set_xlabel(f"Battery capacity [Ah, {batt['cells_series']}S]")
ax.set_ylabel("Endurance [min, reserve held back]")
ax.set_title("Endurance vs pack capacity (mass-coupled)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_endurance_vs_capacity.png", dpi=140)
plt.show()
